## Realized volatility Estimation

This notebook implements the Two Scale Realized Volatility Estimator of Ait Sahalia (2005)

In [7]:
import numpy as np
import matplotlib.pyplot as plt

In [8]:
def simulate_hf_data(n_ticks, true_volatility, noise_std):
    dt = 1.0 / n_ticks
    
    true_returns = np.random.normal(0, true_volatility * np.sqrt(dt), n_ticks)
    true_price = np.cumsum(true_returns)
    
    noise = np.random.normal(0, noise_std, n_ticks)
    
    observed_price = true_price + noise
    
    return true_price, observed_price

def calculate_tsrv(observed_price, K):
    n = len(observed_price)
    
    returns_all = np.diff(observed_price)
    rv_all = np.sum(returns_all**2)
    
    rv_subgrids = []

    for i in range(K):
        price_sub = observed_price[i::K]
        returns_sub = np.diff(price_sub)
        rv_subgrids.append(np.sum(returns_sub**2))
        
    rv_avg = np.mean(rv_subgrids)
    
    n_bar = (n - K + 1) / K 
    
    tsrv_estimate = rv_avg - (n_bar / n) * rv_all
    
    return tsrv_estimate



In [9]:
np.random.seed(571)

# Parameters
N_TICKS = 23400          # 1 tick per second for 6.5 hours
TRUE_SIGMA = 0.015       # True daily volatility (1.5%)
TRUE_VARIANCE = TRUE_SIGMA**2 
NOISE_STD = 0.002        # Microstructure noise standard deviation
K_GRIDS = 60             # Sub-sample roughly every 60 seconds

# Generate data
true_prices, obs_prices = simulate_hf_data(N_TICKS, TRUE_SIGMA, NOISE_STD)

# Calculate naive Realized Variance on all ticks
naive_rv = np.sum(np.diff(obs_prices)**2)

# Calculate TSRV
tsrv = calculate_tsrv(obs_prices, K_GRIDS)

# Output Results
print(f"Target True Variance:      {TRUE_VARIANCE:.6f}")
print(f"Naive Realized Variance:   {naive_rv:.6f}  <-- Biased (Blows up due to noise)")
print(f"TSRV Estimate:             {tsrv:.6f}  <-- Corrected (Recovers true variance)")

Target True Variance:      0.000225
Naive Realized Variance:   0.186148  <-- Biased (Blows up due to noise)
TSRV Estimate:             0.000254  <-- Corrected (Recovers true variance)
